> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 02. 색공간과 검출 표시 — 60분

**예상 시간: 60분**

이 노트북은 영상 처리를 처음 배우는 비전공 학부생을 기준으로 합니다. 설명을 외우기보다 **입력 → 배열 → 변환 → 저장 → 재검증** 순서를 따라가세요.

목표는 BGR과 HSV를 구분하고 빨간 픽셀 mask에서 큰 영역만 찾아 원본 복사본에 사각형·label을 그리는 것입니다.

필요한 실행·저장 규칙은 이 노트북에서 다시 제공합니다. [06.OpenCV README](README.md)와 [용어 찾아보기](GLOSSARY.md)는 추가 참고 자료입니다. TODO는 JupyterLab 파일 편집기에서 저장하고, 미완성 검증이 실패하는 것은 정상입니다. 참고 정답 셀은 강사가 안내하는 시점에 펼칩니다.

수업 경로: 색·좌표 15분 → mask 10분 → TODO 25분 → 저장·해석 10분.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

특정 색을 찾는 과정은 원본을 선택 지도인 mask로 바꾸고 그 결과를 복사본 위에 표시하는 일입니다. 빨강은 HSV Hue의 양 끝에 걸치므로 두 구간을 따로 찾은 뒤 합쳐야 합니다. 같은 물체라도 밝기가 바뀌면 규칙의 결과가 달라질 수 있습니다.

### 전체 과정에서 내 코드가 하는 일

```text
원본 BGR
  ↓
HSV
  ↓
낮은/높은 Hue mask
  ↓
OR 결합
  ↓
면적 조건·box
  ↓
복사본 overlay
  ↓
밝기 조건 비교
```

색 mask와 표시 TODO를 구현하고 원본 불변을 검사합니다. 원본·mask·overlay 세 그림에서 선택한 픽셀과 그린 사각형의 의미를 구분합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 빨간 물체에 사각형이 없으면 색 변환 코드가 틀렸나요?

**설명:** mask에 포함된 픽셀이 있는지, 연결 영역 면적이 기준을 넘는지 순서대로 확인합니다. 색 선택과 면적 필터는 별도 단계입니다. 밝기 비교의 참고 알고리즘 결과와 자기 TODO 검증도 구분합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**OpenCV 준비:** Ollama·ROS node는 필요 없습니다. 첫 Python 셀의 `prepare_assets()`가 합성 입력을 준비하고 검증합니다. 입력은 `Data/images/06.OpenCV/sample_scene.png`와 `Data/videos/06.OpenCV/sample_motion.avi`, 출력은 `Data/outputs/06.OpenCV`입니다. 01 노트북을 먼저 실행해 만든 결과가 없어도 시작할 수 있습니다. GUI·USB 카메라는 로컬 컨테이너에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[01_image_video_io.ipynb의 1절](01_image_video_io.ipynb)에 이미지 배열과 입출력이 있습니다. 필요한 내용은 `imread`의 컬러 결과가 **uint8 `(높이,너비,3)` BGR 배열**, 픽셀 접근은 `[y,x]`, 읽기 실패는 `None`, 저장은 `imwrite` 성공 후 다시 읽어 확인한다는 것입니다. 여기서는 첫 셀의 `prepare_assets()`와 `load_image()`로 입력을 새로 준비하므로 01의 gray 파일은 필요 없습니다. `image.copy()`는 원본과 별개로 표시할 배열을 만듭니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **수업 핵심 TODO와 「입력 조건 비교」** → [3일차 슬라이드 17쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s013) · [배포 PDF 17쪽](../슬라이드/3일차.pdf#page=17)에서 23쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.

자습 morphology 구간을 건너뛴 뒤 마지막 밝기 조건 비교 활동까지 마칩니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 원본을 보존하고 변환·영상 저장 결과를 다시 엽니다.
- **단원 마지막 연결 질문:** 같은 작업대 대상의 밝기 또는 크기만 바꿔 mask가 달라지는지 비교합니다. 색 임계값으로 충분한 조건과 어려운 조건을 한 가지씩 적습니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib, json, subprocess, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '06.OpenCV/common').is_dir())
COURSE = ROOT / '06.OpenCV'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
import cv2, numpy as np
from common.vision import SAMPLE_IMAGE, SAMPLE_VIDEO, OUTPUT_DIR, load_image, prepare_assets, probe_video, save_image
from common.contracts import check
asset_report = prepare_assets()
print('Python:', sys.executable)
print('OpenCV:', cv2.__version__, 'NumPy:', np.__version__)
print(json.dumps(asset_report, ensure_ascii=False, indent=2))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 같은 세 숫자도 색공간에 따라 뜻이 다릅니다

OpenCV BGR의 `(0,0,255)`는 빨강입니다. MediaPipe나 웹의 RGB에 같은 세 숫자를 주면 파랑 의미가 됩니다. `cvtColor`는 channel 순서를 바꾸거나 HSV·grayscale 같은 다른 표현으로 계산합니다.

HSV는 색의 종류 Hue, 선명함 Saturation, 밝기 Value를 나눕니다. OpenCV 8-bit Hue는 원형인 색상각을 0–179에 담아 **빨강이 0 근처와 179 근처 두 구간에 걸칩니다.** 한 구간만 검사하면 일부 빨강을 놓칠 수 있습니다.


In [ ]:
# 구분: 수업 핵심 셀
samples_bgr = np.array([[[0,0,255], [0,255,0], [255,0,0], [90,90,90]]], dtype=np.uint8)
samples_hsv = cv2.cvtColor(samples_bgr, cv2.COLOR_BGR2HSV)
for bgr, hsv in zip(samples_bgr[0], samples_hsv[0]):
    print('BGR', bgr.tolist(), '→ HSV', hsv.tolist())


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 2. mask에서 box까지

<!-- procedure:lesson-22-a33e24:start -->
![색 규칙에서 원본 overlay까지](../교재그림/절차/lesson-22.svg)

mask는 선택 지도, box는 요약 위치입니다. 원본을 보존하여 결과와 입력을 나란히 비교합니다.
<!-- procedure:lesson-22-a33e24:end -->


1. `inRange`로 조건을 만족하는 픽셀을 255, 나머지를 0인 mask로 만듭니다.
2. 두 red mask를 `bitwise_or`로 합칩니다.
3. `findContours`로 이어진 영역의 경계를 찾습니다.
4. `contourArea`가 너무 작은 noise를 버립니다.
5. `boundingRect`가 `(x,y,width,height)`를 만듭니다.
6. 입력을 보존하려고 `image.copy()`에 `rectangle`과 `putText`를 그립니다.

배열 접근은 `[y,x]`, drawing 좌표는 `(x,y)`입니다. 오른쪽 아래를 `x+width-1`로 그리면 mask box의 마지막 픽셀에 맞습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

### 실행 전 예측

아래 합성 장면에는 큰 빨간 사각형, 파란 사각형, 매우 작은 빨간 noise가 있습니다. `min_area=100`이면 box는 몇 개여야 할까요? 파란 영역의 mask 값은 0과 255 중 무엇이어야 할까요?


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 수업 핵심 TODO — [color_overlay.py](exercises/color_overlay.py)

수정 파일: [exercises/color_overlay.py](exercises/color_overlay.py)

두 Hue 구간을 합친 binary red mask와 원본을 보존한 annotation, x 좌표 순서 box 목록을 반환하세요. `min_area`가 음수면 잘못된 계약이므로 거부합니다.

JupyterLab 파일 편집기에서 TODO를 구현하고 **저장** 한 뒤 아래 검증을 실행합니다. 검증은 저장된 모듈을 새 이름으로 다시 읽으므로 일반적으로 kernel 재시작은 필요 없습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `segment_red_and_draw(image, min_area=100.0)`의 image는 BGR uint8 `(H,W,3)`입니다. `(mask, annotated, boxes)`를 반환하며 mask는 `(H,W)`의 0/255 uint8, annotated는 같은 크기의 새 BGR 배열, boxes는 `(x,y,width,height)` 튜플 목록입니다. 영역이 없으면 boxes=`[]`입니다.
**작성 흐름 힌트:** 입력·min_area 검사 → HSV 변환 → H=0–10과 170–179, S=100–255, V=80–255 두 inRange를 OR → 외곽 contour 추출 → contourArea가 min_area 이상인 것의 boundingRect 수집 → `(x,y)` 순 정렬 → 복사본에 초록 사각형과 `RED` 표시.
**주의할 경계:** mask는 전체 빨간 영역이며 작은 contour를 제외하는 것은 boxes·annotation 단계입니다. contourArea는 mask의 255 pixel 개수와 항상 같지 않습니다. 원본에 직접 그리지 않습니다.
**설명할 지점:** “Hue가 원형이라 빨강은 양 끝 구간을 합칩니다. 반환 mask와 사각형이 담당하는 정보를 구분하세요.”
**구현에 필요한 세부 조건:** starter에 `cv2`를 import합니다. image는 uint8 3채널 배열, `min_area<0`은 `ValueError`입니다. `findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)`의 contour에서 `contourArea>=min_area`만 고릅니다. mask는 원래 두 threshold의 OR 결과를 반환하고 작은 점 제거로 바꾸지 않습니다. 정렬은 x, 같은 x이면 y 오름차순입니다. `check('color_overlay','exercise')`는 큰 빨강 box `(15,20,51,41)`, Hue 양 구간과 S/V 경계, 면적 등호·box 정렬, 파랑 제외·원본 보존·annotation을 확인합니다. 아래 저장 예제는 자신의 저장된 함수를 다시 읽습니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE/'exercises/color_overlay.py').read_text())


In [ ]:
# 구분: 수업 핵심 셀
check('color_overlay', 'exercise')
print('learner_verification=PASS')


> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

### 참고 정답

강사가 안내하는 시점에 정답과 해설을 비교합니다. 정답은 별도 파일이며 exercise를 덮어쓰지 않습니다. 구현이 한 글자씩 같을 필요는 없고 입력·출력·오류 처리 계약이 같으면 됩니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE/'solutions/color_overlay.py').read_text())


### 정답 해설 — 입력부터 결과까지

**색 선택과 표시 분리:** BGR→HSV 변환 후 빨강의 Hue 양 끝인 0–10과 170–179를 각각 mask로 만들고 OR로 합칩니다. S·V 하한도 적용하므로 같은 빨강이라도 어두우면 선택되지 않을 수 있습니다.

contour 면적이 min_area보다 작으면 건너뜁니다. 남은 bounding box를 x·y 순으로 정렬해 반환 순서를 고정하고, 원본의 copy에 초록 사각형·글자를 그립니다. contour 면적과 mask의 켜진 pixel 개수는 같은 값이라고 가정하지 않습니다.

**경계:** 면적이 기준과 정확히 같으면 `< min_area`가 거짓이라 포함됩니다. `(x,y,w,h)`의 마지막 pixel은 `(x+w-1,y+h-1)`입니다.

**확인 질문:** 표시한 영상에서 다시 색을 판정하면 원래 없던 글자·테두리가 어떤 영향을 줄 수 있나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/color_overlay.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""02_colors_drawing 참고 정답."""

import cv2
import numpy as np


def segment_red_and_draw(
    image: np.ndarray, min_area: float = 100.0
) -> tuple[np.ndarray, np.ndarray, list[tuple[int, int, int, int]]]:
    if not isinstance(image, np.ndarray) or image.dtype != np.uint8:
        raise ValueError("image는 uint8 numpy.ndarray여야 합니다.")
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError("image shape은 (height, width, 3)이어야 합니다.")
    if min_area < 0:
        raise ValueError("min_area는 0 이상이어야 합니다.")
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    # OpenCV uint8 Hue는 0~179이며 빨강은 원형 색상환의 양 끝에 걸칩니다.
    low_red = cv2.inRange(hsv, (0, 100, 80), (10, 255, 255))
    high_red = cv2.inRange(hsv, (170, 100, 80), (179, 255, 255))
    mask = cv2.bitwise_or(low_red, high_red)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # annotation만 복사본에 그립니다. 작은 영역도 원래 binary mask에는 남습니다.
    result = image.copy()
    boxes = []
    for contour in contours:
        if cv2.contourArea(contour) < min_area:
            continue
        x, y, width, height = cv2.boundingRect(contour)
        boxes.append((x, y, width, height))
    # contour 탐색 순서에 의존하지 않도록 x, y 순으로 반환을 고정합니다.
    boxes.sort(key=lambda box: (box[0], box[1]))
    for x, y, width, height in boxes:
        cv2.rectangle(result, (x, y), (x + width - 1, y + height - 1), (0, 255, 0), 2)
        cv2.putText(result, "RED", (x, max(18, y - 6)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (0, 255, 0), 2, cv2.LINE_AA)
    return mask, result, boxes
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check('color_overlay', 'solution')
print('solution_verification=PASS')


In [ ]:
# 구분: 수업 핵심 셀
from common.contracts import _load
segment_red_and_draw = _load('color_overlay', 'exercise').segment_red_and_draw
image = load_image()
before = image.copy()
mask, overlay, boxes = segment_red_and_draw(image, min_area=100)
assert np.array_equal(image, before)
save_image(OUTPUT_DIR/'02_red_mask.png', cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))
save_image(OUTPUT_DIR/'02_red_overlay.png', overlay)
print('mask shape/values:', mask.shape, np.unique(mask).tolist())
print('boxes:', boxes)
print('saved:', OUTPUT_DIR/'02_red_overlay.png')

# 저장·검증한 배열을 PNG로 표시합니다. 카메라나 모델을 새로 실행하지 않습니다.
import base64
import html
from IPython.display import HTML, display

def show_image_comparison(images):
    cards = []
    for pixels, title in images:
        ok, encoded = cv2.imencode('.png', pixels)
        assert ok, '비교 영상의 PNG 인코딩 실패'
        uri = 'data:image/png;base64,' + base64.b64encode(encoded.tobytes()).decode('ascii')
        cards.append(f'<figure style="margin:8px;flex:1;min-width:220px">'
                     f'<img src="{uri}" style="width:100%;height:280px;object-fit:contain" '
                     f'alt="{html.escape(title)}"><figcaption>{html.escape(title)}</figcaption></figure>')
    display(HTML('<div style="display:flex;flex-wrap:wrap">' + ''.join(cards) + '</div>'))

show_image_comparison([(image, '원본'), (mask, '빨강 mask'), (overlay, '학생 함수의 표시 결과')])


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 결과를 읽는 기준

mask가 이진값인지, 빨간 대표 픽셀은 선택되고 파란 대표 픽셀은 제외되는지, 작은 noise가 box 목록에서 제외되는지를 각각 확인합니다. box 수만 같아도 엉뚱한 위치를 찾을 수 있으므로 좌표도 검사합니다. annotation은 사람이 원인을 확인하는 자료이지 mask 정확도를 자동 보장하는 점수는 아닙니다.


> **구분: 수업 후 복습·자습** — 핵심 완료 뒤 원리와 확장을 확인합니다.

## 자습 — 조명, morphology와 임계값

고정 HSV threshold는 그림자·화이트밸런스·반사에서 달라집니다. 실제 적용에서는 대표 환경의 labeled data로 false positive/negative를 측정해야 합니다. `morphologyEx`의 opening은 작은 점을 줄이고 closing은 작은 구멍을 메울 수 있지만 실제 작은 객체도 지울 수 있습니다. 처리 전후를 모두 저장하고 kernel 크기를 근거 없이 키우지 않습니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 완료 질문

1. OpenCV에서 빨간색 tuple은 왜 `(0,0,255)`인가요?
2. 빨강 Hue 범위가 두 구간인 이유는 무엇인가요?
3. 원본에 바로 rectangle을 그리지 않고 copy를 만드는 이유는 무엇인가요?
4. box가 보인다는 사실만으로 실제 객체 인식 모델이라고 결론내릴 수 없는 이유는 무엇인가요?


> **구분: 수업 핵심**

## 입력 조건 비교 — 같은 색 규칙이 언제 실패할까 / 정리 시간 중 5분

기존 합성 입력과 같은 그룹의 밝기 변형을 사용합니다. 아래는 준비된 참고 알고리즘의 비교이며 자신의 TODO 검증을 대신하지 않습니다. 밝기만 바꾸고 같은 mask 규칙으로 얻은 pixel 수를 비교합니다. 실제 조명·반사를 완전히 재현한 실험은 아닙니다.


In [ ]:
from solutions.color_overlay import segment_red_and_draw
condition_source = load_image()
condition_original = condition_source.copy()
condition_rows = []
condition_dir = ROOT/'Data/outputs/course-review/06.OpenCV'
condition_dir.mkdir(parents=True, exist_ok=True)
for name, factor in [('baseline', 1.0), ('dim', 0.15)]:
    variant_image = np.clip(condition_source.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    condition_mask, condition_overlay, condition_boxes = segment_red_and_draw(variant_image, min_area=100)
    save_image(condition_dir/f'{name}.png', condition_overlay)
    condition_rows.append({'condition': name, 'factor': factor,
        'source_group': 'same-synthetic-image', 'red_pixels': int(np.count_nonzero(condition_mask)),
        'boxes': len(condition_boxes)})
assert np.array_equal(condition_source, condition_original)
(condition_dir/'conditions.json').write_text(json.dumps(condition_rows, indent=2) + '\n')
print(condition_rows)
print('같은 대상인데 결과가 달라졌다면 입력 조건과 임계값 중 무엇을 먼저 점검할까요?')

brightness_images = []
for name, title in [('baseline', '기준 밝기'), ('dim', '어둡게 바꾼 조건')]:
    saved_overlay = cv2.imread(str(condition_dir/f'{name}.png'))
    assert saved_overlay is not None
    brightness_images.append((saved_overlay, title))
show_image_comparison(brightness_images)




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 빨간 영역 표시 결과와 밝기를 바꾼 조건 비교; 자습 morphology 제외

[3일차 슬라이드 17쪽](https://docs.google.com/presentation/d/1dmIJ53VWrgEbYRjZqCYmUiRc9kOc_i7DZMNx6Y4a-sk/edit#slide=id.d3_s013) · [배포 PDF 17쪽](../슬라이드/3일차.pdf#page=17)로 돌아가 **17–23쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

